# Final Project
A diffusion model is a generative machine learning model that learns to denoise data step-by-step.
It does two stages:

*(1)* Forward Diffusion

It gradually adds Gaussian noise to an HR image (or meteorological field), until the image becomes almost pure noise.

*(2)* Reverse Diffusion (Learning the Physics of the Data)

The model trains to reverse this process:

## STEPS
STEP 1: Take HR (2.5 km) data  
STEP 2: Apply degradation operator D → create LR (7.5 km)  
STEP 3: Train diffusion model on (LR → HR) pairs  
STEP 4: Use trained model on new LR data to generate super-resolution outputs


Degradation operator D: Gaussian blur + block-average downsampling.
Converts high-resolution (2.5 km) data into low-resolution (7.5 km) data.


 Parameters
   
    ds : xr.Dataset
        High-resolution dataset (2.5 km grid).
    factor : int, default=3
        Downsampling factor (7.5 / 2.5 = 3).
    sigma_px : float, default=0.8
        Gaussian blur radius in pixels. 0.8–1.0 works well.
    vars_intensive : list of str, optional
        List of variable names to process ( ["WINDGUST", "TEMP", "WSPD10"]).
        If None, all variables in ds will be processed.
    yname, xname : str, optional
        Names of spatial dimensions. Automatically detected if None.

 Returns
    
    xr.Dataset
        Coarsened dataset at lower resolution (e.g., 7.5 km).

sigma_px is the standard deviation (σ) of the Gaussian blur kernel, measured in pixels (grid cells).
So when we call 
ndi.gaussian_filter(data, sigma=0.8)
the filter smooths the data with a Gaussian whose spread is 0.8 grid cells wide.

Physical interpretation for your grid σ=0.8 pixels≈0.8×2.5=2.0km

That means the blur smooths over roughly a 2 km radius, mimicking how a 7.5 km model would not “see” details smaller than a few kilometers.

Without this blur, if you simply take every 3rd grid cell (downsampling factor = 3), high-frequency details would alias — small-scale variations fold into larger-scale ones and create unrealistic patterns.

The Gaussian filter acts as a low-pass (anti-alias) filter, removing features smaller than your new grid spacing before coarsening.

So your degradation operator is: 𝐷(𝑥𝐻𝑅)=Downsample(GaussianBlur(𝑥𝐻𝑅,𝜎=0.8))

How to choose σ:

σ ≈ 0.8–1.0 px → good starting range for mild smoothing (keeps patterns realistic).

σ < 0.5 px → almost no blur, aliasing may appear.

σ > 1.5 px → too smooth, removes too much small-scale detail (diffusion model learns an easier but less realistic task).


You can visualize the effect of different 𝜎

In [1]:
import matplotlib.pyplot as plt
plt.imshow(ndi.gaussian_filter(image, sigma=0.8))  # try 0.5, 1.0, 2.0


NameError: name 'ndi' is not defined

In [4]:
import xarray as xr
import numpy as np
import scipy.ndimage as ndi

def apply_degradation_operator_D(
    ds, 
    factor=3, 
    sigma_px=0.8, 
    vars_intensive=None, 
    yname=None, 
    xname=None
):

    # ------------------------------------------------------------
    # Detect coordinate names
    # ------------------------------------------------------------
    if yname is None:
        yname = "south_north" if "south_north" in ds.dims else "y"
    if xname is None:
        xname = "west_east" if "west_east" in ds.dims else "x"
    ds = ds.rename({yname: "y", xname: "x"})

    # ------------------------------------------------------------
    # Variables to process
    # ------------------------------------------------------------
    if vars_intensive is None:
        vars_intensive = list(ds.data_vars.keys())

    # ------------------------------------------------------------
    # Gaussian blur + coarsen operator
    # ------------------------------------------------------------
    def blur_then_blockmean(da):
        def _blur(arr):
            fill = np.where(np.isfinite(arr), arr, np.nanmedian(arr))
            return ndi.gaussian_filter(fill, sigma=sigma_px, mode="nearest")

        blurred = xr.apply_ufunc(
            _blur, da,
            input_core_dims=[["y", "x"]],
            output_core_dims=[["y", "x"]],
            vectorize=True, dask="parallelized"
        )

        return blurred.coarsen(y=factor, x=factor, boundary="trim").mean()

    # ------------------------------------------------------------
    # Apply to selected variables
    # ------------------------------------------------------------
    degraded = xr.Dataset()
    for v in vars_intensive:
        degraded[v] = blur_then_blockmean(ds[v])

    # ------------------------------------------------------------
    # Create coarsened coordinate grid
    # ------------------------------------------------------------
    degraded = degraded.assign_coords(
        y = ds["y"].coarsen(y=factor, boundary="trim").mean(),
        x = ds["x"].coarsen(x=factor, boundary="trim").mean()
    )

    degraded.attrs["degradation_operator"] = f"Gaussian σ={sigma_px}, factor={factor}"
    return degraded

print ('done')

done


In [ ]:
# Load your HR dataset
ds_hr = xr.open_dataset("RTMA_2&5km.nc")

# Define main variables (intensive: mean-like, not accumulations)
vars_main = ["WINDGUST", "TEMP", "WSPD10"]

# Apply degradation operator D
ds_lr = apply_degradation_operator_D(ds_hr, factor=3, sigma_px=0.8, vars_intensive=vars_main)

# Save LR version
ds_lr.to_netcdf("LR_7&5km_D_operator.nc")

print(ds_lr)



Now, when we have LR data, next step is "preprocessing":

- extract variables from xarray
- stack them as channels
- normalize each variable separately
- possibly crop or pad to match patch size